# 2.5D Deformable Attention Diagnostics

This notebook reconstructs the exact model trained by
`train_accelerate_25d.py` for run `20260710_191946_pfgm_fir_25d`, loads its
best validation checkpoint, and inspects the learned 3D deformable attention.

The main distinction from the 2D notebook is explicit throughout: tensors are
`[B,C,D,H,W]`, each offset group predicts `(dx,dy,dz)`, and attention keys live
on a downsampled 3D K/V grid. Every 2D heatmap below therefore states which
depth plane it represents. Query sampling is capped when attention weights are
recomputed, because PyTorch SDPA intentionally does not expose its matrix.

Diagnostics included:

- strict checkpoint and finite-parameter checks;
- held-out 2.5D input reconstruction using the training loader;
- per-module offset magnitude, saturation, boundary clipping, and group diversity;
- depth displacement, in-plane vector fields, sampled-query attention entropy,
  head diversity, and query-centric key maps;
- AdaLN-Zero gate activity across noise radii;
- one-step PFGM++ denoising, MRI/mask ablations, optional saliency;
- TensorBoard training/validation curves and an automated health summary.

## 1. Setup

The architecture values below come from the run's saved `hparams.yml` and
checkpoint tensor shapes, not from whichever values currently happen to be
in the training script. On CUDA
the notebook uses the trained 256 x 256 field of view. CPU mode uses a centered
64 x 64 crop so structural and numerical checks remain practical.

In [ ]:
%matplotlib inline
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

PROJECT_DIR = Path('/mnt/nvme/lloyd/Swin-pfgm/Swin_pfgm')
sys.path.insert(0, str(PROJECT_DIR))

from tools.deformable_attention_25d_diagnostics import (
    AttentionCapture3D,
    attention_module_names,
    attention_statistics,
    build_model,
    edm_precondition,
    gate_statistics,
    group_offset_similarity,
    load_validation_patch,
    masked_metrics,
    offset_statistics,
    pfgmpp_perturb,
    run_forward_diagnostics,
    scaled_offsets,
    spatial_center_crop,
    to_device_batch,
    training_scalars,
)

RUN_DIR = PROJECT_DIR / 'outputs_accelerate_25d/20260710_191946_pfgm_fir_25d'
CHECKPOINT = RUN_DIR / 'checkpoints/best/model.safetensors'
EVENT_FILE = next((RUN_DIR / 'logs/pfgm_fir_25d').glob('events.out.tfevents.*'))
DATA_DIR = Path('/mnt/nvme/lloyd/DL project CT-MRI/pfgm_brain_train_n')

DEVICE = torch.device(os.environ.get(
    'ATTN_DEVICE', 'cuda:0' if torch.cuda.is_available() else 'cpu'
))
DIAGNOSTIC_SIZE = 256 if DEVICE.type == 'cuda' else 64
PATIENT_INDEX = 0
SIGMA_PROBE = 1.0
SIGMA_LEVELS = [80.0, 10.0, 1.0, 0.1, 0.01]
MAX_ATTENTION_QUERIES = 256
SELECTED_ATTENTION = 'encoder_stages.2.0.attn_blocks.0.attn'
RUN_ABLATIONS = True
RUN_SALIENCY = DEVICE.type == 'cuda'

torch.manual_seed(42)
print('Device:', DEVICE)
print('Diagnostic field of view:', DIAGNOSTIC_SIZE)
print('Checkpoint:', CHECKPOINT)

## 2. Checkpoint and training history

`model.safetensors` is the online model saved when stochastic validation loss
improved. The training loop used EMA weights for generated validation images,
then restored online weights before `save_state`; the separate EMA shadow was
not registered with Accelerate. Thus this file is correctly called the best
validation-loss checkpoint, but it is not the EMA snapshot used for logged
generation images.

There is also an important mask detail: `use_body_mask: true` enables mask
loading, loss masking, and deformable-attention K/V masking, but the model was
constructed with `in_channels=2`. The checkpoint confirms that its input conv
takes noisy CT and MRI only; the mask is not a third image channel.

In [ ]:
model, checkpoint_report = build_model(CHECKPOINT, DEVICE)
module_names = attention_module_names(model)

print('Strict checkpoint load: PASS')
print('Mask concatenated at model input:', model.use_mask_channel)
for key, value in checkpoint_report.items():
    print(f'{key:>30}: {value}')
print('\nDeformable attention modules:')
for name in module_names:
    marker = '  < selected' if name == SELECTED_ATTENTION else ''
    print(' ', name + marker)

assert checkpoint_report['nonfinite_parameters'] == 0
assert len(module_names) == 10

In [ ]:
history = training_scalars(EVENT_FILE)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for tag, ax, log_y in [
    ('train/epoch_loss', axes[0], True),
    ('val/loss', axes[1], True),
    ('val_gen/psnr', axes[2], False),
]:
    series = history[tag]
    ax.plot(series['step'] + 1, series['value'], linewidth=1.4)
    ax.set(title=tag, xlabel='epoch', ylabel=tag.split('/')[-1])
    if log_y:
        ax.set_yscale('log')
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

val = history['val/loss']
best_index = int(np.argmin(val['value']))
psnr = history['val_gen/psnr']
best_psnr_index = int(np.argmax(psnr['value']))
print(f"Best val/loss: {val['value'][best_index]:.6f} at epoch {val['step'][best_index] + 1}")
print(f"Final val/loss: {val['value'][-1]:.6f}")
print(f"Best logged val_gen/psnr: {psnr['value'][best_psnr_index]:.3f} dB "
      f"at epoch {psnr['step'][best_psnr_index] + 1}")
print(f"Final logged val_gen/psnr: {psnr['value'][-1]:.3f} dB")

## 3. Held-out 2.5D sample

The patient split, RAS orientation, four-slice extraction, in-plane resizing,
clamping, and binary mask handling match `loader_25d.py`. All four depth slices
are shown before attention is inspected.

In [ ]:
sample, sample_meta = load_validation_patch(DATA_DIR, patient_index=PATIENT_INDEX)
sample_for_model = spatial_center_crop(sample, DIAGNOSTIC_SIZE)
batch = to_device_batch(sample_for_model, DEVICE)

print(sample_meta)
print({key: tuple(value.shape) for key, value in batch.items()})
print('MRI range:', float(batch['mr'].min()), float(batch['mr'].max()))
print('CT range:', float(batch['ct'].min()), float(batch['ct'].max()))
print('Mask occupancy:', float(batch['mask'].mean()))

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for depth in range(4):
    for row, (key, cmap) in enumerate([('mr', 'gray'), ('ct', 'gray'), ('mask', 'gray')]):
        axes[row, depth].imshow(sample_for_model[key][0, depth].numpy(), cmap=cmap)
        axes[row, depth].set_title(f'{key.upper()} depth={depth}')
        axes[row, depth].axis('off')
plt.tight_layout()
plt.show()

## 4. Instrumented forward pass

One PFGM++ perturbation is generated exactly as in training. Hooks retain all
raw offset fields and normalized Q/K for the selected stage. No model source is
modified, and the hooks are removed immediately after this pass.

In [ ]:
sigma = torch.full((1,), SIGMA_PROBE, device=DEVICE)
noisy = pfgmpp_perturb(batch['ct'], sigma, augmented_dim=128, seed=42)

with AttentionCapture3D(model, SELECTED_ATTENTION) as capture:
    with torch.no_grad():
        prediction = edm_precondition(
            model, noisy, batch['mr'], sigma, batch['mask'], sigma_data=0.45
        )

noisy_metrics = masked_metrics(noisy, batch['ct'], batch['mask'])
prediction_metrics = masked_metrics(prediction, batch['ct'], batch['mask'])
offset_rows = offset_statistics(model, capture.offsets)
attn_report = attention_statistics(
    model, capture, max_queries=MAX_ATTENTION_QUERIES, seed=42
)

print('Output finite:', bool(torch.isfinite(prediction).all()))
print('Noisy metrics:', noisy_metrics)
print('Denoised metrics:', prediction_metrics)
print('Captured offset modules:', len(capture.offsets))
print('Selected query shape:', attn_report['query_shape'])
print('Selected K/V shape:', attn_report['kv_shape'])
print('Sampled-query normalized entropy:', attn_report['entropy_mean'])

In [ ]:
center = batch['ct'].shape[2] // 2
images = [
    batch['mr'][0, 0, center],
    batch['ct'][0, 0, center],
    noisy[0, 0, center],
    prediction[0, 0, center],
    (prediction - batch['ct']).abs()[0, 0, center],
]
titles = ['MRI', 'Clean CT', 'PFGM++ perturbed', 'One-step denoised', '|error|']
cmaps = ['gray', 'gray', 'gray', 'gray', 'magma']

fig, axes = plt.subplots(1, 5, figsize=(19, 4))
for ax, image, title, cmap in zip(axes, images, titles, cmaps):
    artist = ax.imshow(image.detach().cpu(), cmap=cmap)
    ax.set_title(title)
    ax.axis('off')
    if title == '|error|':
        plt.colorbar(artist, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 5. Offset health across the network

Offsets are reported in K/V-grid voxels. With `offset_range_factor=8`, each
component is bounded to plus or minus 8 K/V voxels before the deformed grid is
clamped. `preclamp_oob_fraction` is therefore important: large learned offsets
near volume boundaries may all collapse onto the same clamped edge coordinate.

In [ ]:
header = (
    f"{'module':58} {'K/V':>11} {'|dx|':>7} {'|dy|':>7} {'|dz|':>7} "
    f"{'|d|':>7} {'sat%':>7} {'OOB%':>7}"
)
print(header)
print('-' * len(header))
for row in offset_rows:
    short = row['module'].replace('.attn_blocks.', '.attn.')
    print(
        f"{short:58} {str(row['kv_shape']):>11} "
        f"{row['dx_abs_mean']:7.3f} {row['dy_abs_mean']:7.3f} "
        f"{row['dz_abs_mean']:7.3f} {row['magnitude_mean']:7.3f} "
        f"{100 * row['saturated_fraction']:7.2f} "
        f"{100 * row['preclamp_oob_fraction']:7.2f}"
    )

## 6. Spatial and depth offset visualization

For each of the eight groups, the first row overlays in-plane `(dx,dy)` vectors
and displacement magnitude on the corresponding MRI plane. The second row
shows signed `dz`: blue samples toward lower depth indices and red toward
higher depth indices. Vector lengths are converted from K/V voxels to input
display pixels.

In [ ]:
selected_module = dict(model.named_modules())[SELECTED_ATTENTION]
offset_info = scaled_offsets(selected_module, capture.offsets[SELECTED_ATTENTION])
offset_voxel = offset_info['voxel'][0]
depth_k, height_k, width_k = offset_info['kv_shape']
depth_plane = depth_k // 2
input_depth = batch['mr'].shape[2]
input_plane = round(depth_plane * (input_depth - 1) / max(depth_k - 1, 1))
mri_plane = batch['mr'][0, 0, input_plane].detach().cpu().numpy()

fig, axes = plt.subplots(2, selected_module.n_groups, figsize=(24, 7))
y_grid, x_grid = np.mgrid[0:DIAGNOSTIC_SIZE:complex(height_k),
                          0:DIAGNOSTIC_SIZE:complex(width_k)]
step = max(1, height_k // 12)
for group in range(selected_module.n_groups):
    dx = offset_voxel[group, 0, depth_plane].numpy()
    dy = offset_voxel[group, 1, depth_plane].numpy()
    dz = offset_voxel[group, 2, depth_plane].numpy()
    magnitude = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2)

    axes[0, group].imshow(mri_plane, cmap='gray')
    mag_up = F.interpolate(
        torch.from_numpy(magnitude)[None, None],
        size=mri_plane.shape, mode='bilinear', align_corners=False
    )[0, 0].numpy()
    axes[0, group].imshow(mag_up, cmap='magma', alpha=0.48)
    axes[0, group].quiver(
        x_grid[::step, ::step], y_grid[::step, ::step],
        dx[::step, ::step] * DIAGNOSTIC_SIZE / width_k,
        dy[::step, ::step] * DIAGNOSTIC_SIZE / height_k,
        color='cyan', angles='xy', scale_units='xy', scale=1, width=0.004,
    )
    axes[0, group].set_title(f'group {group}: H/W')
    axes[0, group].axis('off')

    image = axes[1, group].imshow(
        dz, cmap='coolwarm', vmin=-selected_module.offset_range_factor,
        vmax=selected_module.offset_range_factor,
    )
    axes[1, group].set_title(f'group {group}: dz, mean={dz.mean():.2f}')
    axes[1, group].axis('off')

fig.suptitle(
    f'{SELECTED_ATTENTION} | K/V depth={depth_plane}, MRI depth={input_plane}', y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
component_means = offset_voxel.abs().mean(dim=(2, 3, 4)).numpy()
signed_depth = offset_voxel[:, 2].mean(dim=(2, 3)).numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(selected_module.n_groups)
width = 0.25
for component, label in enumerate(['|dx|', '|dy|', '|dz|']):
    axes[0].bar(x + (component - 1) * width, component_means[:, component],
                width=width, label=label)
axes[0].set(xticks=x, xlabel='offset group', ylabel='mean K/V-voxel displacement',
            title='Axis usage by offset group')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.25)

for group in range(selected_module.n_groups):
    axes[1].plot(np.arange(depth_k), signed_depth[group], marker='o', label=f'G{group}')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set(xlabel='K/V depth plane', ylabel='mean signed dz (K/V voxels)',
            title='Learned through-plane routing')
axes[1].legend(ncol=4, fontsize=8)
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Attention weights in 3D

Q and K are captured after RMSNorm and attention is recomputed for at most 256
queries. The key-importance map is the mean attention received by each K/V
location. Each panel is one K/V depth plane, overlaid on its nearest MRI input
slice. Normalized entropy near 1 means diffuse weights; that is not automatically
a failure because deformable offsets already route the K/V samples before QK
weighting. Offset and weight diagnostics must be read together.

In [ ]:
key_importance = attn_report['key_importance'].mean(dim=0)
key_volume = key_importance.reshape(*attn_report['kv_shape']).numpy()
valid_volume = attn_report['valid_keys'].reshape(*attn_report['kv_shape']).numpy()
depth_k, height_k, width_k = attn_report['kv_shape']

fig, axes = plt.subplots(1, depth_k, figsize=(5 * depth_k, 4))
axes = np.atleast_1d(axes)
for depth in range(depth_k):
    input_plane = round(depth * (input_depth - 1) / max(depth_k - 1, 1))
    base = batch['mr'][0, 0, input_plane].detach().cpu().numpy()
    importance = F.interpolate(
        torch.from_numpy(key_volume[depth])[None, None],
        size=base.shape, mode='bilinear', align_corners=False
    )[0, 0].numpy()
    lo, hi = np.percentile(importance, [2, 98])
    display = np.clip((importance - lo) / (hi - lo + 1e-12), 0, 1)
    axes[depth].imshow(base, cmap='gray')
    axes[depth].imshow(display, cmap='hot', alpha=0.58)
    axes[depth].set_title(
        f'K/V depth={depth} -> MRI depth={input_plane}\n'
        f'valid keys={valid_volume[depth].mean():.1%}'
    )
    axes[depth].axis('off')
plt.suptitle('Mean key importance across sampled queries and heads')
plt.tight_layout()
plt.show()

print('Entropy mean/min:', attn_report['entropy_mean'], attn_report['entropy_min'])
print('Valid K/V keys:', int(attn_report['valid_keys'].sum()), '/', len(attn_report['valid_keys']))

In [ ]:
# Exact attention distribution for a center-volume query.
q = capture.selected['q'][0]
k = capture.selected['k'][0]
depth_q, height_q, width_q = attn_report['query_shape']
query_coord = (depth_q // 2, height_q // 2, width_q // 2)
query_index = ((query_coord[0] * height_q) + query_coord[1]) * width_q + query_coord[2]
logits = torch.matmul(q[:, query_index:query_index + 1], k.transpose(-1, -2))
logits = logits / math.sqrt(q.shape[-1])
logits[..., ~attn_report['valid_keys']] = float('-inf')
query_attention = logits.softmax(dim=-1)[..., 0, :].mean(dim=0)
query_volume = query_attention.reshape(*attn_report['kv_shape']).numpy()

fig, axes = plt.subplots(1, depth_k, figsize=(5 * depth_k, 4))
axes = np.atleast_1d(axes)
for depth in range(depth_k):
    input_plane = round(depth * (input_depth - 1) / max(depth_k - 1, 1))
    base = batch['mr'][0, 0, input_plane].detach().cpu().numpy()
    heat = F.interpolate(
        torch.from_numpy(query_volume[depth])[None, None], size=base.shape,
        mode='bilinear', align_corners=False
    )[0, 0].numpy()
    lo, hi = np.percentile(heat, [2, 98])
    axes[depth].imshow(base, cmap='gray')
    axes[depth].imshow(np.clip((heat - lo) / (hi - lo + 1e-12), 0, 1),
                       cmap='hot', alpha=0.58)
    axes[depth].set_title(f'center query -> K/V depth={depth}')
    axes[depth].axis('off')
plt.suptitle(f'Query-centric 3D attention at query coordinate {query_coord}')
plt.tight_layout()
plt.show()

## 8. Head and offset-group diversity

Eight attention heads and eight offset groups are distinct concepts. In this
run there is one head per offset group, but the code still computes diversity
from separate signals: attention key patterns for heads and `(dx,dy,dz)` fields
for groups. Similarity is calculated after removing each map's mean so a shared
near-uniform baseline does not make every pair look artificially identical.

In [ ]:
group_report = group_offset_similarity(attn_report['offset_info'])
head_similarity = attn_report['head_similarity'].numpy()
group_similarity = group_report['matrix'].numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, matrix, title in [
    (axes[0], head_similarity, 'Head key-pattern cosine similarity'),
    (axes[1], group_similarity, 'Offset-group cosine similarity'),
]:
    artist = ax.imshow(matrix, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set(xticks=range(8), yticks=range(8), xlabel='index', ylabel='index', title=title)
    plt.colorbar(artist, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print('Head off-diagonal mean:', attn_report['head_similarity_off_diagonal'])
print('Offset-group off-diagonal mean:', group_report['off_diagonal_mean'])

## 9. Radius conditioning through AdaLN-Zero

Radius conditioning does not change the offsets directly. The Fourier radius
embedding drives six AdaLN vectors in each block; `gate_msa` controls how much
deformable attention enters the residual stream and `gate_mlp` controls the
MLP branch. Nonzero, radius-dependent gates show that the zero-initialized
branches became active during training.

In [ ]:
gates = gate_statistics(model, SIGMA_LEVELS, DEVICE)
gate_modules = sorted(set(row['module'] for row in gates))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for module_name in gate_modules:
    rows = [row for row in gates if row['module'] == module_name]
    sigmas = [row['sigma'] for row in rows]
    axes[0].plot(sigmas, [row['gate_msa_abs_mean'] for row in rows], alpha=0.7)
    axes[1].plot(sigmas, [row['gate_mlp_abs_mean'] for row in rows], alpha=0.7)
for ax, title in zip(axes, ['Attention gate |gate_msa|', 'MLP gate |gate_mlp|']):
    ax.set_xscale('log')
    ax.set(xlabel='sigma', ylabel='mean absolute gate', title=title)
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(f"{'sigma':>8} {'mean |MSA gate|':>18} {'mean |MLP gate|':>18}")
for sigma_value in SIGMA_LEVELS:
    rows = [row for row in gates if row['sigma'] == sigma_value]
    print(f"{sigma_value:8.3g} "
          f"{np.mean([row['gate_msa_abs_mean'] for row in rows]):18.5f} "
          f"{np.mean([row['gate_mlp_abs_mean'] for row in rows]):18.5f}")

## 10. Conditioning and mask ablations

These checks hold the noisy CT fixed and change only MRI or the attention mask.
The relative output change is a sensitivity measure, not a quality metric. A
near-zero MRI response would suggest that the conditional model ignores MRI;
a mask response shows that sampled-background suppression affects inference.

In [ ]:
if RUN_ABLATIONS:
    forward_report = run_forward_diagnostics(
        model, batch, sigma_value=SIGMA_PROBE, seed=42
    )
    print('Noisy:', forward_report['noisy_metrics'])
    print('Prediction:', forward_report['prediction_metrics'])
    print('Zero-MRI relative change:', forward_report['zero_mri_relative_change'])
    print('All-ones-mask relative change:', forward_report['open_mask_relative_change'])

    center = batch['ct'].shape[2] // 2
    normal = forward_report['prediction'][0, 0, center]
    zero_mri = forward_report['prediction_zero_mri'][0, 0, center]
    open_mask = forward_report['prediction_open_mask'][0, 0, center]
    displays = [
        (normal, 'normal'),
        ((normal - zero_mri).abs(), '|normal - zero MRI|'),
        ((normal - open_mask).abs(), '|normal - all-ones mask|'),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, (image, title) in zip(axes, displays):
        artist = ax.imshow(image.detach().cpu(), cmap='gray' if title == 'normal' else 'magma')
        ax.set_title(title)
        ax.axis('off')
        if title != 'normal':
            plt.colorbar(artist, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
else:
    forward_report = None
    print('Ablations disabled. Set RUN_ABLATIONS=True in the setup cell.')

## 11. Optional MRI saliency by depth

This computes gradient-times-input attribution for all four MRI slices. It is
disabled by default on CPU because backward through ten 3D attention modules is
slow. Saliency answers a different question from attention weights: which MRI
voxels most affect denoising error for this sample and radius.

In [ ]:
if RUN_SALIENCY:
    model.zero_grad(set_to_none=True)
    mri_grad = batch['mr'].detach().clone().requires_grad_(True)
    prediction_grad = edm_precondition(
        model, noisy.detach(), mri_grad, sigma, batch['mask'], sigma_data=0.45
    )
    valid = batch['mask'].clamp(0, 1)
    loss = (((prediction_grad - batch['ct']) ** 2) * valid).sum() / valid.sum().clamp_min(1)
    loss.backward()
    saliency = (mri_grad.grad * mri_grad).abs().mean(dim=1)[0].detach().cpu()

    fig, axes = plt.subplots(1, saliency.shape[0], figsize=(4 * saliency.shape[0], 4))
    axes = np.atleast_1d(axes)
    for depth, ax in enumerate(axes):
        base = batch['mr'][0, 0, depth].detach().cpu()
        heat = saliency[depth]
        ax.imshow(base, cmap='gray')
        ax.imshow(heat / heat.quantile(0.98).clamp_min(1e-12), cmap='hot', alpha=0.55)
        ax.set_title(f'MRI saliency depth={depth}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Saliency disabled. Set RUN_SALIENCY=True to run the backward pass.')

## 12. Automated health summary

Thresholds below are guards against clear implementation or checkpoint
failures, not universal performance targets. In particular, attention entropy
and diversity should be compared across patients and radii before drawing
scientific conclusions.

In [ ]:
mean_offset = float(np.mean([row['magnitude_mean'] for row in offset_rows]))
mean_saturation = float(np.mean([row['saturated_fraction'] for row in offset_rows]))
mean_oob = float(np.mean([row['preclamp_oob_fraction'] for row in offset_rows]))
mean_gate = float(np.mean([row['gate_msa_abs_mean'] for row in gates]))
rmse_ratio = prediction_metrics['rmse'] / noisy_metrics['rmse']

checks = {
    'checkpoint loaded strictly': not checkpoint_report['missing_keys'] and not checkpoint_report['unexpected_keys'],
    'all checkpoint parameters finite': checkpoint_report['nonfinite_parameters'] == 0,
    'all model outputs finite': bool(torch.isfinite(prediction).all()),
    'all ten deformable modules executed': len(capture.offsets) == 10,
    'offset predictor is nontrivial': mean_offset > 1e-3,
    'offset tanh is not broadly saturated': mean_saturation < 0.25,
    'AdaLN attention gates are active': mean_gate > 1e-3,
    'single-step denoising improves RMSE': rmse_ratio < 1.0,
}

for label, passed in checks.items():
    print(f"[{'PASS' if passed else 'CHECK'}] {label}")
print()
print(f'Mean offset magnitude: {mean_offset:.4f} K/V voxels')
print(f'Mean saturation fraction: {mean_saturation:.2%}')
print(f'Mean pre-clamp out-of-bounds fraction: {mean_oob:.2%}')
print(f'Mean sampled-query entropy: {attn_report["entropy_mean"]:.4f}')
print(f'Head centered similarity: {attn_report["head_similarity_off_diagonal"]:.4f}')
print(f'Group centered similarity: {group_report["off_diagonal_mean"]:.4f}')
print(f'One-step denoised/noisy RMSE ratio: {rmse_ratio:.4f}')

## Interpretation notes

- A high attention entropy can coexist with useful deformable routing. Inspect
  offset magnitude/diversity and the AdaLN gate before calling a block inactive.
- Depth offsets are real in this architecture even though `kv_stride=2` maps to
  `(1,2,2)`. The stride preserves depth; it does not disable `dz` prediction.
- Pre-clamp out-of-bounds sampling is a particularly relevant diagnostic with
  only four input slices and `offset_range_factor=8`. Reference grids include
  their boundary endpoints, so some OOB coordinates are expected whenever a
  boundary offset points outward. Treat a high fraction as a prompt to inspect
  clipping by axis, not as an implementation failure by itself.
- One held-out patch is a functional diagnostic. For performance claims, loop
  over multiple validation patients/radii and aggregate these reports.
- The best checkpoint is not EMA. Reproducing logged EMA generation exactly
  would require saving/loading the EMA shadow state in future runs.